# Day 18: Git, CI/CD and Docker
**Azure Repos, Azure Pipelines YAML, build and test automation, and Dockerfiles**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup
Git is installed in Colab. Docker is not, so Docker cells are for VS Code with Docker Desktop (or an Azure Pipelines agent).

In [ ]:
# ===== Option A: Google Colab (active) =====
!git --version
!pip install -q pyyaml

# ===== Option B: VS Code / local =====
# Install Git (git-scm.com) and Docker Desktop, then check in the terminal:
#   git --version
#   docker --version
import os, shutil
shutil.rmtree("tasks-api", ignore_errors=True)
os.makedirs("tasks-api/app", exist_ok=True)
os.chdir("tasks-api")
print("Working in", os.getcwd())

## 2. A small app to version

In [ ]:
%%writefile app/main.py
from fastapi import FastAPI

app = FastAPI(title="Tasks API")
tasks = []

@app.get("/tasks")
def list_tasks():
    return tasks

In [ ]:
%%writefile requirements.txt
fastapi
uvicorn

## 3. First commits

In [ ]:
# ===== Option A: Google Colab (active) =====
!git init -q -b main
!git config user.name "Learner" && git config user.email "learner@example.com"
!git add .
!git commit -q -m "Initial FastAPI app"
!git log --oneline

# ===== Option B: VS Code terminal =====
# Same commands without the "!" at the start.

## 4. A feature branch and merge

In [ ]:
!git switch -q -c feature/health-check
!printf '\n@app.get("/health")\ndef health():\n    return {"status": "ok"}\n' >> app/main.py
!git add app/main.py && git commit -q -m "Add health check endpoint"
!git switch -q main
!git merge -q feature/health-check
!git branch -d feature/health-check
!git log --oneline

## 5. A merge conflict, resolved
Two branches change the same line. We merge, see the conflict, and resolve it.

In [ ]:
!echo 'greeting = "Hello"' > greeting.py && git add . && git commit -q -m "Add greeting"
!git switch -q -c feature/japanese
!echo 'greeting = "Konnichiwa"' > greeting.py && git commit -q -am "Japanese greeting"
!git switch -q main
!echo 'greeting = "Namaste"' > greeting.py && git commit -q -am "Hindi greeting"
!git merge feature/japanese || true
!cat greeting.py

In [ ]:
# Resolve: keep both greetings, remove the markers
open("greeting.py", "w").write('greetings = ["Namaste", "Konnichiwa"]\n')
!git add greeting.py && git commit -q -m "Resolve greeting conflict: keep both"
!git log --oneline --graph

### TODO 1: Create a branch `feature/count`, add a `GET /tasks/count` endpoint, commit, and merge it into main.

In [ ]:
# TODO: write your code here

## 6. Push to Azure Repos (VS Code or Colab with access)
Create an empty repo in Azure DevOps first. Never paste tokens into a shared notebook.

In [ ]:
# ===== Option B: VS Code terminal (recommended) =====
# git remote add origin https://dev.azure.com/<org>/<project>/_git/tasks-api
# git push -u origin main
# git switch -c feature/priority-filter
# ...commit...
# git push -u origin feature/priority-filter
# Then open a pull request in Azure DevOps: Repos > Pull requests > New pull request
print("Remote commands are listed above")

## 7. Docker files

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app/ ./app

RUN useradd -m appuser
USER appuser

EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

In [ ]:
%%writefile .dockerignore
.git
__pycache__/
*.pyc
.venv/
tests/
*.md

In [ ]:
# Read the Dockerfile and explain each instruction
explain = {"FROM": "base image", "WORKDIR": "folder inside the image", "COPY": "copy files in",
           "RUN": "run a command at build time", "USER": "switch user", "EXPOSE": "document the port",
           "CMD": "command when the container starts"}
for line in open("Dockerfile"):
    if line.strip():
        word = line.split()[0]
        print(f"{line.strip():<60} <- {explain.get(word, '')}")

In [ ]:
# ===== Option B: VS Code with Docker Desktop =====
# docker build -t tasks-api:1.0 .
# docker run -d -p 8000:8000 --name tasks tasks-api:1.0
# curl http://localhost:8000/health
# docker logs tasks
# docker stop tasks && docker rm tasks
print("Run the Docker commands above in your terminal")

## 8. A multi-stage pipeline

In [ ]:
%%writefile azure-pipelines.yml
trigger:
  - main

pr:
  - main

variables:
  imageName: tasks-api

stages:
  - stage: Test
    displayName: Lint and test
    jobs:
      - job: unit
        pool:
          vmImage: ubuntu-latest
        steps:
          - task: UsePythonVersion@0
            inputs:
              versionSpec: "3.11"
          - script: pip install -r requirements.txt pytest httpx
            displayName: Install
          - script: python -m pytest -q --junitxml=results.xml || echo "add tests in tests/"
            displayName: Run tests
          - task: PublishTestResults@2
            condition: always()
            inputs:
              testResultsFiles: results.xml

  - stage: Docker
    displayName: Build image
    dependsOn: Test
    condition: and(succeeded(), eq(variables['Build.SourceBranch'], 'refs/heads/main'))
    jobs:
      - job: build
        pool:
          vmImage: ubuntu-latest
        steps:
          - script: docker build -t $(imageName):$(Build.BuildId) .
            displayName: docker build

In [ ]:
import yaml
p = yaml.safe_load(open("azure-pipelines.yml"))
for st in p["stages"]:
    jobs = [j["job"] for j in st["jobs"]]
    print(f"Stage {st['stage']:<7} depends on {st.get('dependsOn', '-'):<5} jobs={jobs}")
    if "condition" in st:
        print("   runs when:", st["condition"])

### TODO 2: Add the files to Git with a commit message that explains why, then view the history with `git log --oneline`.

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)